[![Fixel Algorithms](https://fixelalgorithms.co/images/CCExt.png)](https://fixelalgorithms.gitlab.io)

# Deep Learning Methods

## Deep Learning - Vision and Language - Text Visual Search

This notebook builds a text to image search engine with `EmbeddingGemma 2`.  

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 13/10/2026 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2026_02/WorkShop002/0002TextVisualSearch.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Image Processing
import imageio.v3 as iio

# Deep Learning
import torch
from sentence_transformers import SentenceTransformer

# Miscellaneous
import os
import random
import yaml

# Visualization
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython
from IPython.display import display

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

In [ ]:
# Constants

FIG_SIZE_DEF    = (8, 8)
ELM_SIZE_DEF    = 50
CLASS_COLOR     = ('b', 'r')
EDGE_COLOR      = 'k'
MARKER_SIZE_DEF = 10
LINE_WIDTH_DEF  = 2

PROJECT_NAME       = 'FixelCourses'
DATA_FOLDER_NAME   = 'DataSets'
MODELS_FOLDER_NAME = 'Models'
BASE_FOLDER_PATH   = os.getcwd()[:(len(os.getcwd()) - (os.getcwd()[::-1].lower().find(PROJECT_NAME.lower()[::-1])))]
DATA_FOLDER_PATH   = os.path.join(BASE_FOLDER_PATH, DATA_FOLDER_NAME)
MODELS_FOLDER_PATH = os.path.join(BASE_FOLDER_PATH, MODELS_FOLDER_NAME)

## Motivation

Given a folder of images, find those matching a natural language sentence.  
For example, _"A vehicle with two wheels"_.

![](https://i.postimg.cc/YCWMLrRp/image.png)

The query need not contain a class name or a file name.  
The goal is to search by visual content, not by keywords.

* <font color='red'>(**?**)</font> Which computations can be done before the user enters a query?

## A Shared Embedding Space

An _Embedding_ represents an input as a vector.  
A multimodal embedding model maps images and text into a shared space.

The image vectors are calculated before the search.  
At query time, only the text needs to be encoded.  
The query vector is compared with every stored image vector.

Related content should receive higher similarity than unrelated content.  
The result is a ranked list of images, not a generated answer.

* <font color='brown'>(**#**)</font> Image and query embeddings must come from the same model and use the same output dimension.  
  Equal vector dimensions alone do not make embeddings from different models compatible.

## EmbeddingGemma 2 and SigLIP

[EmbeddingGemma 2](https://ai.google.dev/gemma/docs/embeddinggemma/model_card_2) is an open multimodal embedding model built on the Gemma 4 architecture.  

It has a shared backbone to generate representations for _Audio_, _Text_ and _Image_ in a joint _embedding space_.

<!-- ![](https://i.imgur.com/H8yLsPs.png) -->
![](https://i.postimg.cc/JngYKSVs/image.png)



| Property         | SigLIP / SigLIP 2                                    | EmbeddingGemma 2                                     |
|------------------|------------------------------------------------------|------------------------------------------------------|
| Structure        | Separate image and text towers                       | Modality encoders with a shared Gemma based backbone |
| Inputs           | Images and text                                      | Text, images, audio, video and mixed inputs          |
| Text Preparation | Model specific tokenization and templates            | Task prefixes can guide the embedding                |
| Search           | Compare separately calculated image and text vectors | Compare separately calculated image and text vectors |

We use [`google/embeddinggemma-2`](https://huggingface.co/google/embeddinggemma-2) through _Sentence Transformers_.  
The package handles preprocessing, pooling and normalization.  

### Task Specific Embeddings

The same text can have different embeddings for different tasks.  
A short prefix tells the model how the text will be used.  
This changes the input, not the model weights or the embedding dimension.  
Since the convention was used during training while the tasks were aligned, it created a specialized sub paths in the model.

| `prompt_name`        | Intended Use                                | Prefix                                 |
|----------------------|---------------------------------------------|----------------------------------------|
| `SearchQuery`        | A search query                              | `task: search result \| query: `       |
| `Document`           | Text to be searched, including our captions | `title: none \| text: `                |
| `Classification`     | Text classification                         | `task: classification \| query: `      |
| `Clustering`         | Grouping related texts                      | `task: clustering \| query: `          |
| `SentenceSimilarity` | Comparing the meaning of two texts          | `task: sentence similarity \| query: ` |

Passing `prompt_name` to `encode()` adds the prefix automatically.  
We use `SearchQuery` for the query and `Document` for stored captions. Images receive no text prefix.  
A sentence similarity prompt is not a replacement for the retrieval prompts.

* <font color='brown'>(**#**)</font> Changing the task prefix changes the representation. Rebuild affected cached embeddings.
* <font color='brown'>(**#**)</font> The 1st `EmbeddingGemma`model is text only. The 2nd iteration added the _Audio_ and _Image_ modalities.
* <font color='brown'>(**#**)</font> The structure of the model is modular. In this notebook only the _Image_ and _Text_ encoders are utilized.

In [ ]:
# Parameters

# Data
dataDirName     = 'ImgSearchNLP'
lImgFileExt     = ['.jpg', '.jpeg', '.png']
captionFileName = 'Captions.yaml'

# Model
modelId    = 'google/embeddinggemma-2'
runDevice  = 'cuda:0' if torch.cuda.is_available() else 'cpu'
modelDType = torch.float32

# Image Index
batchSize     = 8
indexFileName = 'ImgSearchNLPEmbeddingGemma2.npz'
rebuildIndex  = False

# Caption Index
captionIndexFileName = 'ImgSearchNLPCaptionsEmbeddingGemma2.npz'

# Search
numResults    = 6
captionWeight = 0.35

## The Search Pipeline

1. Dataset: List the image files and inspect a few examples.
2. Image Index: Encode each image once and store its normalized embedding.
3. Search: Encode a sentence, rank the image vectors and display the top results.

Image files stay on disk. Only their embeddings are needed to calculate search scores.  
The model weights remain fixed.

The final section builds a separate caption index and combines the two sources of evidence.  
Keeping the image-only results gives us a baseline for comparison.

* <font color='red'>(**?**)</font> How does the cost of building the index differ from the cost of answering one query?

## Generate / Load Data

We use `FixelCourses/DataSets/ImgSearchNLP`: 5,000 Flickr images in 50 category folders.  
<!-- The files use Flickr Small 400 with the supplied aspect ratio. No cropping is applied.

 - `README.md` records the source pages, authors and licenses.
 - `DownloadState.json` records file metadata and hashes.
 - `Captions.yaml` contains 2-3 descriptions for each image reviewed so far.

The captions were written by an assistant from half-size previews. They are not human-verified annotations.  
Caption coverage is currently partial. The final section reports the actual coverage.

The image-only search does not use category names or captions.  
The final section adds captions as optional hints.

The sorted file list defines the order of rows in the image index.  
Images may be stored directly in the folder or in subfolders. -->

In [ ]:
# Generate / Load Data

dataDirPath = os.path.join(DATA_FOLDER_PATH, dataDirName)
lImgFile = []

for dirPath, _, lFileName in os.walk(dataDirPath):
    for fileName in lFileName:
        if os.path.splitext(fileName)[1].lower() in lImgFileExt:
            imgFilePath = os.path.join(dirPath, fileName)
            lImgFile.append(os.path.relpath(imgFilePath, dataDirPath))

lImgFile.sort()
numSamples = len(lImgFile)

print(f'Number of images: {numSamples}')

### Plot the Data

In [ ]:
# Plot the Data

numPreview  = min(10, numSamples)
numCols     = min(5, numPreview)
numRows     = (numPreview + numCols - 1) // numCols
lPreviewIdx = random.sample(range(numSamples), numPreview)

hF, vHa = plt.subplots(nrows = numRows, ncols = numCols, figsize = (2 * numCols, 2 * numRows), squeeze = False)

for hA in vHa.flat:
    hA.axis('off')

for imgIdx, hA in zip(lPreviewIdx, vHa.flat):
    imgFilePath = os.path.join(dataDirPath, lImgFile[imgIdx])
    tI = iio.imread(imgFilePath)
    hA.imshow(tI)
    hA.set_title(f'Image {imgIdx}')

hF.tight_layout()

## Pre Calculate Image Embeddings

Each image is encoded independently. The result is one vector per image.  
The vectors are normalized to unit L2 norm and stored as rows in `mImgEmb`.  
Row $i$ corresponds to `lImgFile[i]`.

We save the vectors and their file list in a NumPy archive.  
The index is reused when the model settings and image file metadata match.  
Set `rebuildIndex = True` after changing model weights or preprocessing.

### Load the Model

_Sentence Transformers_ provides `encode()` for text and multimodal inputs.  
Images are supplied explicitly under the `image` key. A plain string would be treated as text.  
The package handles the model's image preprocessing.

* <font color='brown'>(**#**)</font> This checkpoint requires Sentence Transformers 6.1 or newer and a Transformers version supporting EmbeddingGemma 2.  
  The first model load downloads the weights. Later loads can reuse the local cache.
* <font color='brown'>(**#**)</font> This notebook uses `float32` for the model and stored embedding vectors.

In [ ]:
# Model

oModel = SentenceTransformer(modelId, device = runDevice,
                             model_kwargs = {'torch_dtype': modelDType},
                             config_kwargs = {'audio_config': None})
oModel.eval()

embDim = oModel.get_embedding_dimension()
if embDim is None:
    raise ValueError('The model does not report an embedding dimension.')

print(f'Embedding dimension: {embDim}')
print(f'Device: {oModel.device}')

In [ ]:
# Load Image Index

indexFilePath = os.path.join(MODELS_FOLDER_PATH, 'TextVisualSearch', indexFileName)
lImgPath     = [os.path.join(dataDirPath, imgFile) for imgFile in lImgFile]
lFileStat    = [os.stat(imgPath) for imgPath in lImgPath]
mFileInfo    = np.array([(oStat.st_size, oStat.st_mtime_ns) for oStat in lFileStat], dtype = np.int64)
mImgEmb      = None

if os.path.isfile(indexFilePath) and not rebuildIndex:
    with np.load(indexFilePath, allow_pickle = False) as dIndex:
        cacheMatches = (
            dIndex['modelId'].item() == modelId
            and dIndex['modelDType'].item() == str(modelDType)
            and dIndex['dataDirPath'].item() == os.path.abspath(dataDirPath)
            and dIndex['lImgFile'].tolist() == lImgFile
            and np.array_equal(dIndex['mFileInfo'], mFileInfo)
        )
        if cacheMatches:
            mCachedEmb = dIndex['mImgEmb'].astype(np.float32, copy = True)
            if (mCachedEmb.shape == (numSamples, embDim)
                    and np.isfinite(mCachedEmb).all()
                    and np.allclose(np.linalg.norm(mCachedEmb, axis = 1), 1.0, atol = 1e-4)):
                mImgEmb = mCachedEmb

print('Loaded the image index.' if mImgEmb is not None else 'Image embeddings need to be calculated.')

In [ ]:
# Build Image Index

if mImgEmb is None:
    mImgEmb = np.empty((numSamples, embDim), dtype = np.float32)

    for batchStart in range(0, numSamples, batchSize):
        batchEnd = min(batchStart + batchSize, numSamples)
        lBatchInputs = [{'image': iio.imread(imgPath)} for imgPath in lImgPath[batchStart:batchEnd]]
        mBatchEmb = oModel.encode(lBatchInputs, batch_size = batchSize, prompt = '',
                                  normalize_embeddings = True, convert_to_numpy = True,
                                  show_progress_bar = False)
        mImgEmb[batchStart:batchEnd, :] = np.asarray(mBatchEmb, dtype = np.float32)
        print(f'Encoded {batchEnd} / {numSamples} images.')

    vImgNorm = np.linalg.norm(mImgEmb, axis = 1, keepdims = True)
    if not np.isfinite(mImgEmb).all() or np.any(vImgNorm == 0):
        raise ValueError('The model returned invalid image embeddings.')
    mImgEmb = mImgEmb / vImgNorm

    os.makedirs(os.path.dirname(indexFilePath), exist_ok = True)
    np.savez_compressed(indexFilePath, mImgEmb = mImgEmb, lImgFile = np.asarray(lImgFile),
                        mFileInfo = mFileInfo, modelId = modelId, modelDType = str(modelDType),
                        dataDirPath = os.path.abspath(dataDirPath))
    print(f'Saved image index: {indexFilePath}')

print(f'Image embedding matrix shape: {mImgEmb.shape}')

## Text to Image Search

A query is a natural language sentence describing the desired visual content.  
We encode it with the model's `SearchQuery` prompt. The package adds the task prefix.  
Image inputs do not receive a text prefix.

Let $\hat{\boldsymbol{q}} \in \mathbb{R}^{D}$ be the normalized query embedding.  
The rows of $\boldsymbol{V} \in \mathbb{R}^{N \times D}$ are the normalized image embeddings.

$$ \boldsymbol{s} = \boldsymbol{V} \hat{\boldsymbol{q}}, \qquad s_i = \hat{\boldsymbol{v}}_i^{T} \hat{\boldsymbol{q}} $$

Sort the scores from highest to lowest and return the corresponding images.  
Only the query is encoded again. The image index is reused.

* <font color='brown'>(**#**)</font> The scores are cosine similarities, not probabilities. A top result is not necessarily a good match.
* <font color='red'>(**?**)</font> What should the search return when none of the images matches the sentence?

In [ ]:
# Encode the Search Query

queryStr = 'A vehicle with two wheels.'
queryStr = queryStr.strip()

if not queryStr:
    raise ValueError('Enter a nonempty search query.')

vQueryEmb = oModel.encode(queryStr, prompt_name = 'SearchQuery',
                          normalize_embeddings = True, convert_to_numpy = True,
                          show_progress_bar = False)
vQueryEmb = np.asarray(vQueryEmb, dtype = np.float32)
queryNorm = np.linalg.norm(vQueryEmb)

if not np.isfinite(vQueryEmb).all() or queryNorm == 0:
    raise ValueError('The model returned an invalid query embedding.')
vQueryEmb = vQueryEmb / queryNorm

print(f'Query: {queryStr}')

### Search Results

Return the images with the highest query similarity.  
The table keeps the file paths, scores and rank. The gallery shows the same order.

* <font color='brown'>(**#**)</font> A search result is an existing image from the dataset. The model does not generate a new image.

In [ ]:
# Rank and Display Search Results

if not isinstance(numResults, int) or numResults < 1:
    raise ValueError('numResults must be a positive integer.')

vSimilarity = mImgEmb @ vQueryEmb
vTopIdx     = np.argsort(-vSimilarity, kind = 'stable')[:min(numResults, numSamples)]

dfResults = pd.DataFrame({
    'Rank': np.arange(1, len(vTopIdx) + 1),
    'Image': [lImgFile[imgIdx] for imgIdx in vTopIdx],
    'Cosine Similarity': vSimilarity[vTopIdx],
})
display(dfResults)

numCols = min(3, len(vTopIdx))
numRows = (len(vTopIdx) + numCols - 1) // numCols
hF, vHa = plt.subplots(nrows = numRows, ncols = numCols, figsize = (3 * numCols, 3 * numRows), squeeze = False)

for hA in vHa.flat:
    hA.axis('off')

for rankIdx, (imgIdx, hA) in enumerate(zip(vTopIdx, vHa.flat), start = 1):
    imgFilePath = os.path.join(dataDirPath, lImgFile[imgIdx])
    tI = iio.imread(imgFilePath)
    hA.imshow(tI)
    hA.set_title(f'{rankIdx}. Similarity: {vSimilarity[imgIdx]:0.3f}')

hF.tight_layout()

* <font color='blue'>(**!**)</font> Search for the same concept using two different sentences. Compare the rankings.
* <font color='blue'>(**!**)</font> Describe an object or scene missing from the dataset. Inspect the returned scores.
* <font color='red'>(**?**)</font> Does the highest score guarantee that the image matches every detail in the query?

* <font color='green'>(**@**)</font> Extend the engine to image to image search. Reuse the stored image index.
* <font color='green'>(**@**)</font> Repeat the search with SigLIP 2. Build a separate image index and compare the retrieved results.

## Combined Image and Caption Search

An image contains visual detail. Captions add descriptions of objects, actions and context.  
We keep both representations and combine their matching scores. This is _Late Fusion_.

### Weighted Evidence

Let $\hat{\boldsymbol{q}}$ be the normalized query vector, $\hat{\boldsymbol{v}}_i$ the image vector and $\hat{\boldsymbol{c}}_{ij}$ its caption vectors.  
For an image with $M_i$ captions:

$$ a_i = \hat{\boldsymbol{v}}_i^T \hat{\boldsymbol{q}}, \qquad b_i = \frac{1}{M_i} \sum_{j=1}^{M_i} \hat{\boldsymbol{c}}_{ij}^T \hat{\boldsymbol{q}} $$

We average the caption scores. An extra caption should not add a score simply because it exists.

Text and image similarities can have different baselines. Mixing the raw values may favor one modality.  
We first convert each score vector to percentile ranks: lowest is 0, highest is 1, and ties share their average rank.  
Image ranks use all images. Caption ranks use only images with captions. A single candidate receives 0.5.

Let $r_i^I$ and $r_i^C$ be these rank scores. The combined score is:

$$ s_i = \begin{cases} (1 - \alpha) r_i^I + \alpha r_i^C, & M_i > 0 \\ r_i^I, & M_i = 0 \end{cases}, \qquad 0 \leq \alpha \leq 1 $$

`captionWeight` sets $\alpha$. At 0, the ranking matches the image-only search.  
Images without captions keep their image rank score. They are not assigned a zero caption similarity.

### Pre Calculate Caption Embeddings

Captions are loaded from `Captions.yaml` and encoded with the model's `Document` prompt.  
The query still uses `SearchQuery`. All vectors come from the same model and are L2 normalized.  
A separate cache is reused only when the captions, image order, model, precision and prompt match.

* <font color='brown'>(**#**)</font> Rank scores are not probabilities. They depend on the candidate set and discard absolute score differences.
* <font color='brown'>(**#**)</font> Caption coverage is partial and concentrated in a few categories. This can bias the comparison.  
  Compare both methods on the same candidate images and report caption coverage.
* <font color='brown'>(**#**)</font> Incorrect or incomplete captions can hurt retrieval. Better accuracy is not guaranteed.  
  Tune the weight on held-out queries with known relevant images, using measures such as Precision@K or Recall@K.
* <font color='red'>(**?**)</font> Which queries benefit from captions? Which depend on details that the captions missed?

In [ ]:
# Load Captions

captionFilePath = os.path.join(dataDirPath, captionFileName)
with open(captionFilePath, 'r', encoding = 'utf-8') as hFile:
    dCaptionData = yaml.safe_load(hFile)

if not isinstance(dCaptionData, dict) or not isinstance(dCaptionData.get('images'), dict):
    raise ValueError('The caption YAML must contain an images mapping.')

dCaptionByFile = dCaptionData['images']
lCaptionText   = []
lCaptionImgIdx = []

for imgIdx, imgFile in enumerate(lImgFile):
    imgKey = os.path.normpath(imgFile).replace(os.sep, '/')
    lText = dCaptionByFile.get(imgKey, [])
    if not isinstance(lText, list) or any(not isinstance(text, str) or not text.strip() for text in lText):
        raise ValueError(f'Invalid captions for: {imgFile}')
    lCaptionText.extend(text.strip() for text in lText)
    lCaptionImgIdx.extend([imgIdx] * len(lText))

vCaptionImgIdx = np.asarray(lCaptionImgIdx, dtype = np.int64)
vCaptionCount  = np.bincount(vCaptionImgIdx, minlength = numSamples)
vHasCaption    = vCaptionCount > 0

print(f'Images with captions: {np.count_nonzero(vHasCaption)} / {numSamples}')
print(f'Number of captions: {len(lCaptionText)}')
if not np.any(vHasCaption):
    print('No captions matched the image list. Combined search will use image scores only.')

In [ ]:
# Pre Calculate Caption Embeddings

captionPromptName    = 'Document'
captionIndexFilePath = os.path.join(MODELS_FOLDER_PATH, 'TextVisualSearch', captionIndexFileName)
mCaptionEmb          = None

if lCaptionText:
    if os.path.isfile(captionIndexFilePath) and not rebuildIndex:
        with np.load(captionIndexFilePath, allow_pickle = False) as dIndex:
            cacheMatches = (
                dIndex['modelId'].item() == modelId
                and dIndex['modelDType'].item() == str(modelDType)
                and dIndex['promptName'].item() == captionPromptName
                and dIndex['lImgFile'].tolist() == lImgFile
                and dIndex['lCaptionText'].tolist() == lCaptionText
                and np.array_equal(dIndex['vCaptionImgIdx'], vCaptionImgIdx)
            )
            if cacheMatches:
                mCachedEmb = dIndex['mCaptionEmb'].astype(np.float32, copy = True)
                if (mCachedEmb.shape == (len(lCaptionText), embDim)
                        and np.isfinite(mCachedEmb).all()
                        and np.allclose(np.linalg.norm(mCachedEmb, axis = 1), 1.0, atol = 1e-4)):
                    mCaptionEmb = mCachedEmb
                    print('Loaded the caption index.')

    if mCaptionEmb is None:
        mCaptionEmb = oModel.encode(lCaptionText, batch_size = batchSize, prompt_name = captionPromptName,
                                    normalize_embeddings = True, convert_to_numpy = True,
                                    show_progress_bar = True)
        mCaptionEmb = np.asarray(mCaptionEmb, dtype = np.float32)
        vCaptionNorm = np.linalg.norm(mCaptionEmb, axis = 1, keepdims = True)
        if (mCaptionEmb.shape != (len(lCaptionText), embDim)
                or not np.isfinite(mCaptionEmb).all() or np.any(vCaptionNorm == 0)):
            raise ValueError('The model returned invalid caption embeddings.')
        mCaptionEmb = mCaptionEmb / vCaptionNorm

        os.makedirs(os.path.dirname(captionIndexFilePath), exist_ok = True)
        np.savez_compressed(captionIndexFilePath, mCaptionEmb = mCaptionEmb,
                            lImgFile = np.asarray(lImgFile), lCaptionText = np.asarray(lCaptionText),
                            vCaptionImgIdx = vCaptionImgIdx, modelId = modelId,
                            modelDType = str(modelDType), promptName = captionPromptName)
        print(f'Saved caption index: {captionIndexFilePath}')
else:
    mCaptionEmb = np.empty((0, embDim), dtype = np.float32)

print(f'Caption embedding matrix shape: {mCaptionEmb.shape}')

In [ ]:
# Combine Image and Caption Scores

def ToRankScore( vScore: np.ndarray ) -> np.ndarray:
    if vScore.size < 2:
        return np.full(vScore.shape, 0.5, dtype = np.float32)
    vRank = sp.stats.rankdata(vScore, method = 'average')
    return ((vRank - 1) / (vScore.size - 1)).astype(np.float32)

if not 0 <= captionWeight <= 1:
    raise ValueError('captionWeight must be between 0 and 1.')
if not isinstance(numResults, int) or numResults < 1:
    raise ValueError('numResults must be a positive integer.')

vImageSimilarity   = mImgEmb @ vQueryEmb
vImageRankScore    = ToRankScore(vImageSimilarity)
vCaptionScore      = mCaptionEmb @ vQueryEmb
vCaptionSimilarity = np.full(numSamples, np.nan, dtype = np.float32)
vCaptionRankScore  = np.full(numSamples, np.nan, dtype = np.float32)

if np.any(vHasCaption):
    vCaptionSum = np.bincount(vCaptionImgIdx, weights = vCaptionScore, minlength = numSamples)
    vCaptionSimilarity[vHasCaption] = vCaptionSum[vHasCaption] / vCaptionCount[vHasCaption]
    vCaptionRankScore[vHasCaption] = ToRankScore(vCaptionSimilarity[vHasCaption])

vCombinedScore = vImageRankScore.copy()
vCombinedScore[vHasCaption] = ((1 - captionWeight) * vImageRankScore[vHasCaption]
                               + captionWeight * vCaptionRankScore[vHasCaption])
vCombinedTopIdx = np.argsort(-vCombinedScore, kind = 'stable')[:min(numResults, numSamples)]

vImageOrder = np.argsort(-vImageSimilarity, kind = 'stable')
vImageOnlyRank = np.empty(numSamples, dtype = np.int64)
vImageOnlyRank[vImageOrder] = np.arange(1, numSamples + 1)

print(f'Query: {queryStr}')
print(f'Caption weight: {captionWeight:0.2f}')

In [ ]:
# Display Combined Search Results

lCaptionHint = []
for imgIdx in vCombinedTopIdx:
    vCaptionIdx = np.flatnonzero(vCaptionImgIdx == imgIdx)
    if vCaptionIdx.size:
        bestCaptionIdx = vCaptionIdx[np.argmax(vCaptionScore[vCaptionIdx])]
        lCaptionHint.append(lCaptionText[bestCaptionIdx])
    else:
        lCaptionHint.append('No caption available')

dfCombinedResults = pd.DataFrame({
    'Rank': np.arange(1, len(vCombinedTopIdx) + 1),
    'Image': [lImgFile[imgIdx] for imgIdx in vCombinedTopIdx],
    'Image-Only Rank': vImageOnlyRank[vCombinedTopIdx],
    'Image Rank Score': vImageRankScore[vCombinedTopIdx],
    'Caption Rank Score': vCaptionRankScore[vCombinedTopIdx],
    'Combined Score': vCombinedScore[vCombinedTopIdx],
    'Captions': vCaptionCount[vCombinedTopIdx],
    'Best Caption': lCaptionHint,
})
display(dfCombinedResults)

numCols = min(3, len(vCombinedTopIdx))
numRows = (len(vCombinedTopIdx) + numCols - 1) // numCols
hF, vHa = plt.subplots(nrows = numRows, ncols = numCols, figsize = (3 * numCols, 3 * numRows), squeeze = False)

for hA in vHa.flat:
    hA.axis('off')

for rankIdx, (imgIdx, hA) in enumerate(zip(vCombinedTopIdx, vHa.flat), start = 1):
    imgFilePath = os.path.join(dataDirPath, lImgFile[imgIdx])
    tI = iio.imread(imgFilePath)
    hA.imshow(tI)
    hA.set_title(f'{rankIdx}. Combined: {vCombinedScore[imgIdx]:0.3f}')

hF.tight_layout()